# Sumativa Fase 3 — Semana 2

## Núcleo algorítmico y programación orientada a objetos

**Titulo: Factores sociodemograficos y su asociación con factores de Riesgo Cardiovascular(ENS 2016-2017)**

---

En la Fase 2 se construyó un pipeline que funciona. Este cuaderno lo toma y le
da forma de **sistema**: las mismas operaciones, ahora organizadas en clases que se
pueden probar, reemplazar y reutilizar.

No hay datos nuevos ni pipeline nuevo. **Lo que cambia es la arquitectura del código.**

### Este es el recorrido

| Parte | Contenido | Criterio de la rúbrica que alimenta |
|---|---|---|
| 1 | De funciones a objetos: por qué | Codificación funcional |
| 2 | Los cinco conceptos de la POO | Programación orientada a objetos |
| 3 | La clase `Preprocesador` | Preprocesamiento y transformación |
| 4 | Encapsulamiento: proteger el estado | Programación orientada a objetos |
| 5 | Herencia y polimorfismo | Programación orientada a objetos |
| 6 | El `Pipeline`: componer los pasos | Diseño estructurado |
| 7 | Cohesión y acoplamiento | Diseño estructurado |
| 8 | Validación: casos normales, límite y excepciones | Validación técnica |
| 9 | Recursividad | Diseño estructurado |
| 10 | Eficiencia: medir tiempo y memoria | Eficiencia y optimización |
| 11 | Patrones de diseño | Documentación de arquitectura |
| 12 | Adaptar el cuaderno a su propio conjunto | Codificación funcional |
| 13 | El resultado: código y datos listos para la Fase 4 | Documentación de arquitectura |
| 14 | Ejercicios para su proyecto | Todos |


## Configuración: Descripción de todas las variables de nuestro estudio


In [ ]:
# =====================================================================
# VARIABLES
# =====================================================================

RUTA_DATOS = "data/processed/ens_variables_f1f2.xlsx"

COLUMNA_ID = "IdEncuesta"             # identificador: se eliminó del analisis

# Predictores: caracteristicas sociodemograficas
COLUMNAS_PREDICTORAS_CONTINUAS = ["Edad", "anos_estudio_MINSAL_1", "as27"]
COLUMNAS_PREDICTORAS_NOMINALES = ["Sexo", "Zona"]
COLUMNAS_PREDICTORAS_ORDINALES = ["as28"]

# Resultados: indicadores de riesgo cardiovascular (estudio asociativo,
# no de clasificacion con un solo target - se analizan por separado)
COLUMNAS_RESULTADO_BINARIAS = ["HTA"]
COLUMNAS_RESULTADO_NOMINALES = ["di3", "dis2"]
COLUMNAS_RESULTADO_ORDINALES = ["GPAQ"]
COLUMNAS_RESULTADO_CONTINUAS = ["IMC"]   # confirmar si se incluye

# Diseno muestral: se preservan, no se transforman
COLUMNAS_DISENO_MUESTRAL = ["Fexp_F1F2p_Corr", "Conglomerado", "Estrato"]

SEMILLA = 2026
PERMITIR_DEMOSTRACION = False
# =====================================================================

COLUMNAS_ESPERADAS = (
    [COLUMNA_ID]
    + COLUMNAS_PREDICTORAS_CONTINUAS + COLUMNAS_PREDICTORAS_NOMINALES + COLUMNAS_PREDICTORAS_ORDINALES
    + COLUMNAS_RESULTADO_BINARIAS + COLUMNAS_RESULTADO_NOMINALES + COLUMNAS_RESULTADO_ORDINALES + COLUMNAS_RESULTADO_CONTINUAS
    + COLUMNAS_DISENO_MUESTRAL
)

print("Archivo configurado :", RUTA_DATOS)
print("Columnas esperadas  :", len(COLUMNAS_ESPERADAS))

## Preparación del entorno

In [ ]:
import os
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd

def _encontrar_raiz_temporal(marcador=".git"):
    actual = Path.cwd()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / marcador).exists():
            return carpeta
    raise FileNotFoundError(f"No se encontro '{marcador}' en ningun directorio padre")

RAIZ = _encontrar_raiz_temporal()
sys.path.append(str(RAIZ / "src"))

from carga import encontrar_raiz_proyecto

np.random.seed(SEMILLA)

print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)
print("Raiz   :", RAIZ.name)

## Carga del conjunto desde el archivo externo

In [ ]:
def leer_archivo(ruta):
    """Lee el archivo segun su extension y devuelve un DataFrame."""
    extension = os.path.splitext(ruta)[1].lower()

    if extension in (".csv", ".txt"):
        return pd.read_csv(ruta, sep=None, engine="python", encoding="utf-8")
    if extension in (".xlsx", ".xls"):
        return pd.read_excel(ruta)
    if extension == ".parquet":
        return pd.read_parquet(ruta)

    raise ValueError(
        f"Extension no reconocida: '{extension}'. "
        "Se admiten .csv, .txt, .xlsx, .xls y .parquet."
    )


def verificar_esquema(df, columnas_esperadas):
    """Comprueba que esten todas las columnas declaradas en la configuracion."""
    faltantes = [c for c in columnas_esperadas if c not in df.columns]
    if faltantes:
        raise KeyError(
            f"Faltan columnas declaradas en la configuracion: {faltantes}\n"
            f"Columnas disponibles en el archivo: {list(df.columns)}"
        )
    sobrantes = [c for c in df.columns if c not in columnas_esperadas]
    return {"declaradas": len(columnas_esperadas),
            "en_archivo": df.shape[1],
            "no_declaradas": sobrantes}


def cargar(ruta, columnas_esperadas, permitir_demo=PERMITIR_DEMOSTRACION):
    """Carga el conjunto desde el archivo externo y verifica su esquema."""
    ruta_completa = RAIZ / ruta

    if ruta_completa.exists():
        df = leer_archivo(str(ruta_completa))
        print(f"Archivo leido: {ruta_completa}")
    else:
        raise FileNotFoundError(
            f"No se encontro el archivo: {ruta_completa}\n"
            "Revisen la variable RUTA_DATOS en la celda de configuracion."
        )

    informe = verificar_esquema(df, columnas_esperadas)
    print(f"Forma: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"Esquema verificado: {informe['declaradas']} columnas declaradas, "
          f"{len(informe['no_declaradas'])} no declaradas")
    if informe["no_declaradas"]:
        print("  No declaradas:", informe["no_declaradas"])
    return df


datos = cargar(RUTA_DATOS, COLUMNAS_ESPERADAS)
datos.head()

## El conjunto de datos antes de tocarlo

In [ ]:
def perfilar(df):
    """Resumen rapido: dimensiones, tipos y nulos por columna."""
    print("Dimensiones:", df.shape)
    print("\nTipos de datos:")
    print(df.dtypes.value_counts())
    print("\nValores nulos por columna (solo las que tienen):")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0])
    return df.describe()


perfilar(datos)

## Clase base: Transformador 
Importada desde `src/transformador.py`. Define el contrato que
cumplirán las clases de imputación, codificación y escalamiento.

In [ ]:
from transformador import Transformador

print("Transformador importado correctamente desde src/")